# Buffalo Gals — one opening song

This is **curl_test.ipynb**, with the game added to it.

Run **all cells in order**, from the same project folder as data/, images/, and
keys/service.json. The existing Myra game screen appears immediately. One Lyria
request creates one 30-second instrumental opening song using the three chosen
CSV scenes. The song plays once; character choices and **Play again** do not
generate another song. There is no opening screen, GIF, or video.

Use the same Google project/service account as in your original curl test.
If the Google authentication package is missing, run this in a separate cell
once, then restart the kernel:

~~~python
%pip install google-auth requests
~~~

The player requests automatic playback. If the browser blocks it, press its
Play control. Generation runs in the background so the game remains responsive.


In [1]:
# IMPORTS
import asyncio
import base64
import requests
import pandas as pd
import ipywidgets as widgets

from google.oauth2 import service_account
from google.auth.transport.requests import Request
from PIL import Image
from IPython.display import display, clear_output


ModuleNotFoundError: No module named 'google'

In [ ]:
# GOOGLE AUTHENTICATION — same service account as your curl test
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
# get_audio() obtains/refreshes the access token when it makes the music request.
# The old keys/keys.json cell was unused, so it is not needed here.


In [ ]:
# GENERATE ONE SONG — the full CSV-based prompt replaces the Pokemon name
def get_audio(prompt):
    credentials = creds
    if not credentials.valid:
        credentials.refresh(Request())

    url = (
        "https://aiplatform.googleapis.com/v1beta1/projects/"
        "wpp-cto-smartprod-sandbox/locations/global/interactions"
    )
    payload = {
        "model": "lyria-3-clip-preview",
        "store": False,
        "input": [{"type": "text", "text": prompt}],
    }
    headers = {
        "Authorization": f"Bearer {credentials.token}",
        "Content-Type": "application/json",
    }

    response = requests.post(
        url, headers=headers, json=payload, timeout=(15, 240)
    )
    response.raise_for_status()
    data = response.json()

    # Google documents both response shapes. Find audio by type, not position.
    blocks = list(data.get("outputs") or [])
    for step in data.get("steps") or []:
        if step.get("type") == "model_output":
            blocks.extend(step.get("content") or [])

    for block in blocks:
        if block.get("type") == "audio" and block.get("data"):
            audio_bytes = base64.b64decode(block["data"], validate=True)
            if audio_bytes:
                return audio_bytes

    raise RuntimeError(
        "The music service returned no audio. "
        f"Status: {data.get('status', 'unknown')}"
    )


In [ ]:
# OPENING SONG PROMPT — uses your actual spreadsheet rows
def build_opening_music_prompt():
    scene_details = []

    # All three scenes inspire ONE song, not one song per scene.
    for scene in scenes:
        row = get_interaction(scene["id"])
        scene_details.append(
            f"Characters: {row['actor']} and {row['recipient']}. "
            f"Location: {row['location']}. "
            f"Scene: {row['description']}. "
            f"Outcome: {row['outcome']}."
        )

    return (
        "Create one 30-second instrumental opening theme for a classic "
        "16-bit pixel-art community game. Warm, curious and gently adventurous, "
        "with a memorable soft chiptune melody, rounded bass and light percussion. "
        "Express connection, care and learning to live together. "
        "Instrumental music without singing or spoken narration. "
        "Use these actual game scenes as musical inspiration:\n"
        + "\n".join(scene_details)
    )


In [ ]:
# PLAY THE OPENING SONG WITHOUT BLOCKING THE GAME BUTTONS
async def play_opening_song(prompt, player, status, state, request_audio):
    try:
        # The slow network call runs in a worker thread; widgets update here.
        audio_bytes = await asyncio.to_thread(request_audio, prompt)
        player.value = audio_bytes
        player.layout.display = ""
        status.value = "Opening song — press Play if it doesn't start."
    except asyncio.CancelledError:
        # A notebook rerun may discard this result. It cannot undo a sent request.
        raise
    except Exception as error:
        state["error"] = f"{type(error).__name__}: {error}"
        status.value = "The opening song couldn't load. You can still play."


In [ ]:
# LOAD THE DATASET
interactions_df = pd.read_csv("data/buffalo_gals_community_v1.csv").fillna("")

scenes = [
    {
        "character": "Chickadee",
        "id": "COM038",
        "question": "Will you help Chickadee prepare dinner?",
        "choice": "Help cook",
        "points": 3,
        "image": "chickadee.png",
        "action_image": "chickadeeMyraCookTogether.png"
    },
    {
        "character": "Coyote",
        "id": "COM002",
        "question": "Will you follow Coyote down to the creek?",
        "choice": "Follow Coyote",
        "points": 1,
        "image": "coyote.png",
        "action_image": "theCreek.png"
    },
    {
        "character": "Grandmother",
        "id": "COM053",
        "question": "Will you listen to Grandmother's guidance?",
        "choice": "Listen",
        "points": 2,
        "image": "grandmother.png",
        "action_image": "theCave.png"
    }
]




# FIND EXACTLY ONE RECORD: never invent a missing outcome.
def get_interaction(interaction_id):
    match = interactions_df[
        interactions_df["interaction_id"] == interaction_id
    ]
    if len(match) != 1:
        raise ValueError(f"Expected one row for {interaction_id}; found {len(match)}.")
    return match.iloc[0]

# Check the selected records before displaying the game.
for scene in scenes:
    row = get_interaction(scene["id"])
    participants = f"{row['actor']}; {row['recipient']}".split(";")
    if "Myra" not in [name.strip() for name in participants]:
        raise ValueError(f"Myra is not listed as a participant in {scene['id']}.")

# DISPLAY AN ASSIGNED IMAGE; missing artwork does not stop the game.
def show_image(image_name, size=450):
    if image_name is None:
        return
    path = f"images/{image_name}"
    try:
        with Image.open(path) as img:
            img.thumbnail((size, size), Image.Resampling.NEAREST)
            display(img)
    except OSError:
        print(f"[Image unavailable: {path}]")




# GAME STATE
community_points = 0
completed_ids = []
current_scene = None
phase = "menu"
max_points = sum(scene["points"] for scene in scenes)

# WIDGETS
# Clear the previous player/task when rerunning the game cell.
if "opening_music_task" in globals() and not opening_music_task.done():
    opening_music_task.cancel()

if "opening_music_player" in globals():
    opening_music_player.autoplay = False
    opening_music_player.value = b""
    opening_music_player.close()

opening_music_state = {"error": None}
opening_music_status = widgets.Label(value="Generating the opening song...")
opening_music_player = widgets.Audio(
    format="mp3",
    autoplay=True,
    loop=False,
    controls=True,
    layout=widgets.Layout(display="none"),
)
opening_music_panel = widgets.VBox([
    opening_music_status, opening_music_player
])

output = widgets.Output()
score = widgets.Label()
menu_buttons = []
for scene in scenes:
    menu_buttons.append(widgets.Button(layout=widgets.Layout(width="350px")))

finish_button = widgets.Button(description="Finish exploring")
menu = widgets.VBox(menu_buttons + [finish_button])
action_button = widgets.Button(layout=widgets.Layout(width="220px"))
leave_button = widgets.Button(description="Not now")
choices = widgets.HBox([action_button, leave_button])
explore_button = widgets.Button(description="Let's explore!")
restart_button = widgets.Button(description="Play again")


def update_score():
    score.value = f"Player: Myra | Community Points: {community_points} / {max_points}"


# HOME
def show_menu(b=None):
    global current_scene, phase
    current_scene = None
    phase = "menu"

    for button, scene in zip(menu_buttons, scenes):
        done = scene["id"] in completed_ids
        button.disabled = done
        if done:
            button.description = f"{scene['character']}: completed (+{scene['points']})"
        else:
            button.description = f"{scene['character']}: {scene['choice']} (+{scene['points']})"

    with output:
        clear_output(wait=True)
        print("HI MYRA! \nNOW THAT YOU ARE IN THE DESERT, YOU WILL NEED TO CONNECT WITH THE ANIMAL COMMUNITTY IN ORDER TO SURVIVE. \nCOMMUNITY POINTS WILL GIVE YOU WHAT YOU NEED TO KEEP GOING. \nWHERE WILL YOU GO?")
        print("Choose someone to meet. You can explore in any order.")
        if len(completed_ids) == len(scenes):
            print("\nAll three activities are complete! Finish exploring to see your total.")

    menu.layout.display = "flex"
    choices.layout.display = "none"
    explore_button.layout.display = "none"
    restart_button.layout.display = "none"


# MEET: show choices only after choosing a character.
def on_meet_click(b):
    global current_scene, phase
    if phase != "menu" or b.disabled:
        return

    current_scene = scenes[menu_buttons.index(b)]
    interaction = get_interaction(current_scene["id"])
    phase = "encounter"

    with output:
        clear_output(wait=True)
        print(f"Location: {interaction['location']}")
        show_image(current_scene["image"], 350)
        #print(f"\nYou are Myra, meeting {current_scene['character']}.")
        print(current_scene["question"])

    action_button.description = f"{current_scene['choice']} (+{current_scene['points']})"
    menu.layout.display = "none"
    choices.layout.display = "flex"

    

# ACT: descriptions and outcomes come from the CSV
def on_action_click(b):
    global community_points, phase
    if phase != "encounter":
        return
    if current_scene["id"] in completed_ids:
        return

    interaction = get_interaction(current_scene["id"])
    phase = "result"
    completed_ids.append(current_scene["id"])
    community_points += current_scene["points"]
    update_score()

    with output:
        clear_output(wait=True)
        # Reuse the character portrait when no action image is assigned.
        show_image(current_scene["action_image"] or current_scene["image"], 500)
        print(interaction["description"])
        if interaction["outcome"]:
            print("\nOUTCOME:")
            print(interaction["outcome"])
        print(f"\n+{current_scene['points']} Community Points!")
        print(f"Total so far: {community_points} / {max_points}")
        #print(f"\nSource: {interaction['interaction_id']} | PDF page {interaction['pdf_page']}")

    choices.layout.display = "none"
    explore_button.layout.display = ""


# DECLINE: this option remains available on the menu for a later visit.
def on_leave_click(b):
    global phase
    if phase != "encounter":
        return
    phase = "result"
    with output:
        clear_output(wait=True)
        print("You decide not to take part right now. You can come back later.")
        print("No points added.")
    choices.layout.display = "none"
    explore_button.layout.display = ""


# END: show the points actually earned, including partial playthroughs.
def on_finish_click(b):
    global phase
    if phase != "menu":
        return
    phase = "end"
    with output:
        clear_output(wait=True)
        print("EXPLORATION COMPLETE")
        print(f"\nYou earned {community_points} of {max_points} Community Points!")
        print(f"Activities completed: {len(completed_ids)} / {len(scenes)}\n")
        for scene in scenes:
            earned = scene["points"] if scene["id"] in completed_ids else 0
            print(f"{scene['character']} - {scene['choice']}: +{earned}")

    menu.layout.display = "none"
    choices.layout.display = "none"
    explore_button.layout.display = "none"
    restart_button.layout.display = ""


def on_restart_click(b):
    global community_points, completed_ids
    community_points = 0
    completed_ids = []
    update_score()
    show_menu()



# CONNECT BUTTONS
for button in menu_buttons:
    button.on_click(on_meet_click)

action_button.on_click(on_action_click)
leave_button.on_click(on_leave_click)
explore_button.on_click(show_menu)
finish_button.on_click(on_finish_click)
restart_button.on_click(on_restart_click)

# DISPLAY ONE GAME SCREEN
if "game_screen" in globals():
    game_screen.close()

game_screen = widgets.VBox([
    score, output, menu, choices, explore_button, restart_button,
    opening_music_panel
])
update_score()
show_menu()
from IPython.display import Image as NotebookImage, display

with output:
    display(NotebookImage(filename="images/myra.png", width=150))

display(game_screen)

# ONE OPENING SONG AT STARTUP — no extra screen and no per-character requests
try:
    opening_music_prompt = build_opening_music_prompt()
    opening_music_task = asyncio.create_task(
        play_opening_song(
            opening_music_prompt,
            opening_music_player,
            opening_music_status,
            opening_music_state,
            get_audio,
        )
    )
except Exception as error:
    opening_music_state["error"] = f"{type(error).__name__}: {error}"
    opening_music_status.value = "The opening song couldn't load. You can still play."


## What changed

- get_audio(prompt) now receives a complete musical prompt instead of a Pokemon name.
- build_opening_music_prompt() reads actor, recipient, location, description,
  and outcome from CSV rows **COM038**, **COM002**, and **COM053**
  through your existing scenes list and get_interaction() function.
- The three rows inspire **one opening song**. loop=False plays it once.
- The game still starts with Myra and the same character choices. Music generation
  begins automatically after that screen is displayed.
- Running the game cell again starts a new request. Game choices and Play again do not.
- The audio control sits outside the story output, so changing scenes does not
  clear the player.

### Inspect the generated prompt

In a new cell:

~~~python
print(opening_music_prompt)
~~~

### If the music request fails

If the background music request fails, the game remains usable. Read the error
in a new cell:

~~~python
print(opening_music_state["error"])
~~~

The Google account still needs access to the project and Lyria model from your
original example. No credentials or generated audio are included in this file.

### Implementation references

- [Google Cloud: generate music with Lyria](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/music/generate-music)
- [Google Cloud: Interactions API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/interactions-api)
- [Browser autoplay behavior](https://developer.mozilla.org/en-US/docs/Web/Media/Guides/Autoplay)
